In [0]:
# COMMAND ----------
# ── Install libraries ─────────────────────────────────────────────────────────
%pip install imbalanced-learn==0.12.3 --quiet
dbutils.library.restartPython()

In [0]:
# COMMAND ----------
# ── Constants — identical to Session 1 so everything connects ─────────────────
# Session 2 builds directly on top of what Session 1 produced.
# DO NOT change these values — they must match Session 1 exactly.

CATALOG             = "hive_metastore"
DATABASE            = "mlops_session1"            # same DB as Session 1
EXPERIMENT_S1       = "/Shared/mlops_s1_churn"    # Session 1 experiment (we read from this)
EXPERIMENT_S2       = "/Shared/mlops_s2_churn"    # Session 2 experiment (we write to this)
MODEL_NAME          = "retail_churn_s1"            # same registry model
DELTA_RAW_PATH      = "dbfs:/FileStore/mlops_s1/raw_retail"
DELTA_FEATURES_PATH = "dbfs:/FileStore/mlops_s1/features"
SCORES_PATH_S1      = "dbfs:/FileStore/mlops_s1/churn_scores"
SCORES_PATH_S2      = "dbfs:/FileStore/mlops_s2/shadow_scores"
RANDOM_STATE        = 42
DATA_VERSION        = 1

# Feature column list — must be identical to Session 1
FEATURE_COLS = [
    "recency_days", "frequency", "monetary_total",
    "avg_invoice_value", "avg_unit_price", "unique_products",
    "purchase_days", "total_quantity",
    "avg_items_per_invoice", "spend_per_day",
]
TARGET_COL = "Churn"

print(f"S1 Experiment : {EXPERIMENT_S1}")
print(f"S2 Experiment : {EXPERIMENT_S2}")
print(f"Model name    : {MODEL_NAME}")
print(f"Feature cols  : {len(FEATURE_COLS)}")

In [0]:
# COMMAND ----------
# ── Setup: create S2 folders, set experiment ──────────────────────────────────
import mlflow
from mlflow.tracking import MlflowClient

spark.sql(f"USE {DATABASE}")
dbutils.fs.mkdirs("dbfs:/FileStore/mlops_s2")

mlflow.set_experiment(EXPERIMENT_S2)
mlflow.set_registry_uri("databricks")
client = MlflowClient(registry_uri="databricks")

experiment_s2 = mlflow.get_experiment_by_name(EXPERIMENT_S2)
print(f"S2 Experiment ID: {experiment_s2.experiment_id}")
print(f"Artifact root   : {experiment_s2.artifact_location}")
print()
print("Setup complete. Session 2 experiment is ready.")

In [0]:
# COMMAND ----------
# ── Reload data from Delta (same files, pinned to exact same version as S1) ───
# We never re-download from GitHub.
# We read from Delta Lake using VERSION AS OF — the exact same
# snapshot that Session 1 used. This guarantees reproducibility.

import pandas as pd
import numpy as np
from delta.tables import DeltaTable
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# ── Load features table from Delta (as produced by Session 1) ─────────────────
# This is the exact same feature table — no recomputation needed
dt_feat     = DeltaTable.forPath(spark, DELTA_FEATURES_PATH)
feat_history = dt_feat.history(1).select("version").collect()
FEATURES_VERSION = feat_history[0][0]

df_model = (
    spark.read
    .format("delta")
    .option("versionAsOf", FEATURES_VERSION)
    .load(DELTA_FEATURES_PATH)
    .toPandas()
)

print(f"Loaded features from Delta version {FEATURES_VERSION}")
print(f"Dataset shape : {df_model.shape}")
print(f"Churn rate    : {df_model[TARGET_COL].mean():.1%}")

# ── Reconstruct the same train/test split as Session 1 ────────────────────────
# CRITICAL: same random_state=42 + stratify → identical split every time
X = df_model[FEATURE_COLS]
y = df_model[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# ── SMOTE — training set only, same seed ──────────────────────────────────────
smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"\nTrain (balanced): {len(X_train_bal):,}  |  Test: {len(X_test):,}")
print(f"Train churn rate: {pd.Series(y_train_bal).mean():.1%}  |  Test: {y_test.mean():.1%}")
print("\nData reload complete. Same split as Session 1.")

In [0]:
# COMMAND ----------
# ============================================================
# PART 1: Structured Hyperparameter Tuning
#
# What was done in Session 1 was NOT tuning.
# Manually trying 3 fixed configs is guessing, not searching.
#
# Proper tuning uses:
#   1. A defined search space (ranges, not hand-picked values)
#   2. Cross-validation (not a single train/test split)
#   3. A systematic search strategy
#
# We use sklearn's RandomizedSearchCV:
#   - Samples hyperparameters randomly from distributions
#   - Much more efficient than exhaustive GridSearchCV
#   - 5-fold cross-validation = 5 train/test splits per config
#   - Result: a more honest estimate of generalisation performance
#
# We log every CV trial as a NESTED MLflow run:
#   - Parent run  = the tuning job itself
#   - Child runs  = each individual trial (1 config × 5 folds)
#   This keeps the MLflow UI organised and searchable.
# ============================================================

print("PART 1 — Hyperparameter Tuning")
print("-" * 50)
print("Session 1 approach: try 3 fixed configs manually")
print("Session 2 approach: RandomizedSearchCV over distributions")
print("                    5-fold cross-validation per config")
print("                    Every trial logged as a nested MLflow run")
print()
print("Why RandomizedSearch over GridSearch?")
print("  GridSearch  : tries ALL combinations → n^k runs (exponential)")
print("  RandomSearch: samples n_iter random combos → n_iter runs (linear)")
print("  With 5 parameters × 5 values each = 3125 grid combinations")
print("  RandomSearch: we run just 20 → 95% of GridSearch quality at 0.6% of the cost")

In [0]:
# COMMAND ----------
# ── RandomizedSearchCV for Random Forest with nested MLflow runs ───────────────

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from mlflow.models import infer_signature
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score, average_precision_score
)

# ── Define the search space ───────────────────────────────────────────────────
# scipy.stats distributions allow continuous sampling
from scipy.stats import randint, uniform

RF_SEARCH_SPACE = {
    "n_estimators":      randint(100, 500),          # 100 to 500 trees
    "max_depth":         [None, 5, 8, 10, 15, 20],   # None = unlimited
    "min_samples_leaf":  randint(1, 20),              # 1 to 20
    "min_samples_split": randint(2, 20),              # 2 to 20
    "max_features":      ["sqrt", "log2", 0.5, 0.7], # feature subset size
}

# ── 5-fold stratified cross-validation ───────────────────────────────────────
# Stratified = preserves churn ratio in each fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ── Base estimator with class_weight to handle imbalance ─────────────────────
rf_base = RandomForestClassifier(
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

# ── RandomizedSearchCV ────────────────────────────────────────────────────────
# scoring="roc_auc" optimises for AUC, not accuracy
# n_iter=20 means 20 random hyperparameter combinations tried
rf_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=RF_SEARCH_SPACE,
    n_iter=20,
    scoring="roc_auc",       # maximise AUC across CV folds
    cv=cv,
    refit=True,              # refit best params on full training set
    return_train_score=True,
    random_state=RANDOM_STATE,
    verbose=0,
)

# ── Run inside a PARENT MLflow run ────────────────────────────────────────────
# TEACHING POINT: Nested runs organise tuning in the MLflow UI.
# Parent run  = the whole tuning job (best params, final metrics)
# Child runs  = each individual trial (visible under the parent)

with mlflow.start_run(run_name="rf_randomsearch_parent") as parent_run:

    # Log search space metadata on the parent
    mlflow.log_params({
        "search_strategy":  "RandomizedSearchCV",
        "n_iter":           20,
        "cv_folds":         5,
        "cv_strategy":      "StratifiedKFold",
        "scoring_metric":   "roc_auc",
        "model_type":       "RandomForest",
        "smote_applied":    True,
        "data_version":     DATA_VERSION,
        "features_delta_v": FEATURES_VERSION,
        "feature_count":    len(FEATURE_COLS),
        "random_state":     RANDOM_STATE,
    })
    mlflow.set_tags({
        "session":      "session-2",
        "team":         "mlops-training",
        "model_family": "tree",
        "run_type":     "hyperparameter_search",
    })

    # ── Run the search (trains 20 × 5 = 100 models internally) ────────────────
    print("Running RandomizedSearchCV (20 configs × 5 folds = 100 fits)...")
    rf_search.fit(X_train_bal, y_train_bal)
    print("Search complete.")

    # ── Log each trial as a child run ─────────────────────────────────────────
    cv_results = rf_search.cv_results_
    for i in range(len(cv_results["params"])):
        with mlflow.start_run(run_name=f"rf_trial_{i+1:02d}", nested=True):
            trial_params = cv_results["params"][i]
            trial_params["model_type"]    = "RandomForest_trial"
            trial_params["class_weight"]  = "balanced"
            trial_params["smote_applied"] = True
            mlflow.log_params(trial_params)
            mlflow.log_metrics({
                "cv_mean_auc":  round(cv_results["mean_test_score"][i],  4),
                "cv_std_auc":   round(cv_results["std_test_score"][i],   4),
                "cv_mean_train_auc": round(cv_results["mean_train_score"][i], 4),
                "rank":         int(cv_results["rank_test_score"][i]),
            })
            mlflow.set_tag("run_type", "cv_trial")

    # ── Evaluate the best estimator on the held-out test set ─────────────────
    best_rf     = rf_search.best_estimator_
    y_pred      = best_rf.predict(X_test)
    y_pred_prob = best_rf.predict_proba(X_test)[:, 1]

    final_metrics = {
        "accuracy":      round(accuracy_score(y_test, y_pred), 4),
        "roc_auc":       round(roc_auc_score(y_test, y_pred_prob), 4),
        "avg_precision": round(average_precision_score(y_test, y_pred_prob), 4),
        "f1_score":      round(f1_score(y_test, y_pred, zero_division=0), 4),
        "precision":     round(precision_score(y_test, y_pred, zero_division=0), 4),
        "recall":        round(recall_score(y_test, y_pred, zero_division=0), 4),
        "cv_best_auc":   round(rf_search.best_score_, 4),
        "train_size":    len(X_train_bal),
        "test_size":     len(X_test),
    }
    mlflow.log_metrics(final_metrics)

    # ── Log best hyperparameters on parent run too ────────────────────────────
    best_params = rf_search.best_params_
    mlflow.log_params({f"best_{k}": v for k, v in best_params.items()})

    # ── Log model with signature ──────────────────────────────────────────────
    signature = infer_signature(X_train_bal, best_rf.predict_proba(X_train_bal)[:, 1])
    mlflow.sklearn.log_model(
        sk_model=best_rf,
        name="model",
        signature=signature,
        input_example=X_test.head(5),
    )
    mlflow.log_text("\n".join(FEATURE_COLS), "feature_names.txt")

    rf_tuned_run_id = parent_run.info.run_id

print(f"\nBest hyperparameters found:")
for k, v in best_params.items():
    print(f"  {k:<22}: {v}")
print(f"\nCV best AUC   : {rf_search.best_score_:.4f}")
print(f"Test AUC      : {final_metrics['roc_auc']:.4f}")
print(f"Test F1       : {final_metrics['f1_score']:.4f}")
print(f"\nParent run ID : {rf_tuned_run_id}")
print("\nNavigate to Experiments → mlops_s2_churn → rf_randomsearch_parent")
print("Click the run to expand and see all 20 child trial runs underneath.")

In [0]:
# COMMAND ----------
# ── Tuned XGBoost with cross-validated search ─────────────────────────────────
!pip install xgboost
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score, average_precision_score
)
from scipy.stats import randint, uniform

XGB_SEARCH_SPACE = {
    "n_estimators":     randint(100, 600),
    "max_depth":        randint(3, 10),
    "learning_rate":    uniform(0.01, 0.19),    # 0.01 to 0.20
    "subsample":        uniform(0.6, 0.4),      # 0.6 to 1.0
    "colsample_bytree": uniform(0.5, 0.5),      # 0.5 to 1.0
    "reg_alpha":        uniform(0, 1.0),         # L1 regularisation
    "reg_lambda":       uniform(1.0, 4.0),       # L2 regularisation
    "min_child_weight": randint(1, 10),
}

# For XGBoost: scale_pos_weight handles imbalance without SMOTE
scale_pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())

xgb_base = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    use_label_encoder=False,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

xgb_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=XGB_SEARCH_SPACE,
    n_iter=20,
    scoring="roc_auc",
    cv=cv,
    refit=True,
    return_train_score=True,
    random_state=RANDOM_STATE,
    verbose=0,
)

with mlflow.start_run(run_name="xgb_randomsearch_parent") as parent_run:

    mlflow.log_params({
        "search_strategy":    "RandomizedSearchCV",
        "n_iter":             20,
        "cv_folds":           5,
        "scoring_metric":     "roc_auc",
        "model_type":         "XGBoost",
        "scale_pos_weight":   round(scale_pos_weight, 2),
        "smote_applied":      False,
        "data_version":       DATA_VERSION,
        "features_delta_v":   FEATURES_VERSION,
        "feature_count":      len(FEATURE_COLS),
        "random_state":       RANDOM_STATE,
    })
    mlflow.set_tags({
        "session":      "session-2",
        "team":         "mlops-training",
        "model_family": "boosting",
        "run_type":     "hyperparameter_search",
    })

    print("Running XGBoost RandomizedSearchCV (20 configs × 5 folds)...")
    xgb_search.fit(X_train, y_train)   # XGBoost uses original (non-SMOTE) data
    print("Search complete.")

    cv_results = xgb_search.cv_results_
    for i in range(len(cv_results["params"])):
        with mlflow.start_run(run_name=f"xgb_trial_{i+1:02d}", nested=True):
            trial_params = cv_results["params"][i]
            trial_params["model_type"]         = "XGBoost_trial"
            trial_params["scale_pos_weight"]   = round(scale_pos_weight, 2)
            trial_params["smote_applied"]      = False
            mlflow.log_params(trial_params)
            mlflow.log_metrics({
                "cv_mean_auc":  round(cv_results["mean_test_score"][i],  4),
                "cv_std_auc":   round(cv_results["std_test_score"][i],   4),
                "cv_mean_train_auc": round(cv_results["mean_train_score"][i], 4),
                "rank":         int(cv_results["rank_test_score"][i]),
            })
            mlflow.set_tag("run_type", "cv_trial")

    best_xgb    = xgb_search.best_estimator_
    y_pred      = best_xgb.predict(X_test)
    y_pred_prob = best_xgb.predict_proba(X_test)[:, 1]

    xgb_metrics = {
        "accuracy":      round(accuracy_score(y_test, y_pred), 4),
        "roc_auc":       round(roc_auc_score(y_test, y_pred_prob), 4),
        "avg_precision": round(average_precision_score(y_test, y_pred_prob), 4),
        "f1_score":      round(f1_score(y_test, y_pred, zero_division=0), 4),
        "precision":     round(precision_score(y_test, y_pred, zero_division=0), 4),
        "recall":        round(recall_score(y_test, y_pred, zero_division=0), 4),
        "cv_best_auc":   round(xgb_search.best_score_, 4),
        "train_size":    len(X_train),
        "test_size":     len(X_test),
    }
    mlflow.log_metrics(xgb_metrics)
    mlflow.log_params({f"best_{k}": v for k, v in xgb_search.best_params_.items()})

    signature = infer_signature(X_train, best_xgb.predict_proba(X_train)[:, 1])
    mlflow.sklearn.log_model(
        sk_model=best_xgb,
        name="model",
        signature=signature,
        input_example=X_test.head(5),
    )
    mlflow.log_text("\n".join(FEATURE_COLS), "feature_names.txt")

    xgb_tuned_run_id = parent_run.info.run_id

print(f"\nXGBoost best hyperparameters:")
for k, v in xgb_search.best_params_.items():
    print(f"  {k:<22}: {v:.4f}" if isinstance(v, float) else f"  {k:<22}: {v}")
print(f"\nCV best AUC   : {xgb_search.best_score_:.4f}")
print(f"Test AUC      : {xgb_metrics['roc_auc']:.4f}")
print(f"Test F1       : {xgb_metrics['f1_score']:.4f}")
print(f"\nXGB parent run ID: {xgb_tuned_run_id}")

In [0]:
# COMMAND ----------
# ============================================================
# PART 2: Advanced Experiment Management
#
# Never manually inspect MLflow runs.
# Production teams query the MLflow API programmatically to:
#   - Filter runs by any combination of params, metrics, tags
#   - Build automated promotion decisions
#   - Generate model comparison reports
#   - Audit what was tested before deploying anything
#
# We compare ALL runs across BOTH Session 1 and Session 2.
# This simulates a real team where multiple experiments run
# in parallel and we need to pick the globally best model.
# ============================================================

# ── Query all parent runs (exclude child trial runs) from both sessions ────────
exp_s1 = mlflow.get_experiment_by_name(EXPERIMENT_S1)
exp_s2 = mlflow.get_experiment_by_name(EXPERIMENT_S2)

# Exclude child trial runs by filtering on the run_type tag
all_runs = mlflow.search_runs(
    experiment_ids=[exp_s1.experiment_id, exp_s2.experiment_id],
    filter_string="tags.run_type != 'cv_trial' AND metrics.roc_auc > 0",
    order_by=["metrics.roc_auc DESC"],
)

# ── Build a clean comparison table ────────────────────────────────────────────
compare_cols = [
    "tags.mlflow.runName",
    "metrics.roc_auc",
    "metrics.avg_precision",
    "metrics.f1_score",
    "metrics.precision",
    "metrics.recall",
    "params.model_type",
    "run_id",
    "experiment_id",
]
compare_df = all_runs[compare_cols].copy()
compare_df.columns = ["run_name", "roc_auc", "avg_precision",
                      "f1", "precision", "recall",
                      "model_type", "run_id", "experiment_id"]

# Label which session each run came from
compare_df["session"] = compare_df["experiment_id"].apply(
    lambda eid: "S1" if eid == exp_s1.experiment_id else "S2"
)

print("=" * 80)
print("ALL RUNS — RANKED BY ROC-AUC (Sessions 1 and 2 combined)")
print("=" * 80)
print(compare_df[["session", "run_name", "roc_auc", "avg_precision",
                  "f1", "precision", "recall"]].to_string(index=False))

print("\nKEY OBSERVATION:")
print("  RandomizedSearchCV found better hyperparameters than our manual guesses in S1.")
print("  CV cross-validation gives a more reliable estimate than a single test split.")

In [0]:
# COMMAND ----------
# ── Log confusion matrix and ROC curve as MLflow artifacts ────────────────────
# Plots logged as artifacts are visible in the MLflow UI
# under the Artifacts tab. This makes model debugging visual, not just numeric.

import matplotlib
matplotlib.use("Agg")   # non-interactive backend for Databricks
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.metrics import (
    ConfusionMatrixDisplay, RocCurveDisplay,
    confusion_matrix, roc_curve, auc
)
import tempfile, os

# Pick the best tuned model to plot (the one with highest roc_auc)
best_row = compare_df.iloc[0]
BEST_RUN_ID_S2 = best_row["run_id"]

# Identify which model object corresponds to the best run
if BEST_RUN_ID_S2 == rf_tuned_run_id:
    best_model_s2 = best_rf
    best_name_s2  = "Random Forest (tuned)"
else:
    best_model_s2 = best_xgb
    best_name_s2  = "XGBoost (tuned)"

y_pred_best      = best_model_s2.predict(X_test)
y_pred_prob_best = best_model_s2.predict_proba(X_test)[:, 1]

with tempfile.TemporaryDirectory() as tmpdir:

    # ── Confusion matrix plot ─────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    cm = confusion_matrix(y_test, y_pred_best)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=["No Churn", "Churn"])
    disp.plot(ax=axes[0], colorbar=False)
    axes[0].set_title(f"{best_name_s2} — Confusion Matrix")

    # ── ROC curve plot ────────────────────────────────────────────────────────
    fpr, tpr, _ = roc_curve(y_test, y_pred_prob_best)
    roc_auc_val = auc(fpr, tpr)
    axes[1].plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc_val:.4f}")
    axes[1].plot([0, 1], [0, 1], "k--", lw=1, label="Random classifier")
    axes[1].set_xlabel("False Positive Rate")
    axes[1].set_ylabel("True Positive Rate")
    axes[1].set_title(f"{best_name_s2} — ROC Curve")
    axes[1].legend(loc="lower right")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plot_path = os.path.join(tmpdir, "model_evaluation_plots.png")
    plt.savefig(plot_path, dpi=120, bbox_inches="tight")
    plt.close()

    # ── Feature importance plot ───────────────────────────────────────────────
    if hasattr(best_model_s2, "feature_importances_"):
        fi_vals = best_model_s2.feature_importances_
        fi_df   = pd.DataFrame({"feature": FEATURE_COLS, "importance": fi_vals})
        fi_df   = fi_df.sort_values("importance", ascending=True)

        fig2, ax2 = plt.subplots(figsize=(8, 5))
        ax2.barh(fi_df["feature"], fi_df["importance"], color="steelblue")
        ax2.set_xlabel("Importance")
        ax2.set_title(f"{best_name_s2} — Feature Importances")
        ax2.grid(True, axis="x", alpha=0.3)
        plt.tight_layout()
        fi_path = os.path.join(tmpdir, "feature_importance.png")
        plt.savefig(fi_path, dpi=120, bbox_inches="tight")
        plt.close()

    # ── Log both plots back into the parent MLflow run ────────────────────────
    # We update the existing run — no need to start a new one
    with mlflow.start_run(run_id=BEST_RUN_ID_S2):
        mlflow.log_artifact(plot_path, artifact_path="evaluation_plots")
        if hasattr(best_model_s2, "feature_importances_"):
            mlflow.log_artifact(fi_path, artifact_path="evaluation_plots")
        mlflow.set_tag("plots_logged", "yes")

print(f"Plots logged to run: {BEST_RUN_ID_S2}")
print("Navigate to MLflow → Experiments → mlops_s2_churn → best run → Artifacts tab")
print("to view the confusion matrix, ROC curve, and feature importance.")

In [0]:
# COMMAND ----------
# ============================================================
# PART 3: Champion / Challenger Pattern
#
# In production there is always a model serving
# live traffic — that is the CHAMPION. When a new model is
# trained and passes quality gates, it becomes the CHALLENGER.
# Before the Challenger replaces the Champion, it must prove
# it is actually better.
#
# The promotion rule in this session:
#   New model can be promoted ONLY IF:
#   1. Its test ROC-AUC > current Production model AUC + MIN_IMPROVEMENT
#   2. Its F1 does not drop below the current Production model F1
#   3. It passed a quality gate (AUC > absolute minimum)
#
# This prevents a model that scores marginally better on one
# metric but worse on another from accidentally going live.
# ============================================================

print("PART 3 — Champion / Challenger Pattern")
print("-" * 55)
print("Champion : the model currently serving production traffic")
print("Challenger: a newly trained candidate model")
print()
print("Promotion rules (both must pass):")
print("  1. Challenger AUC > Champion AUC + 0.005 (meaningful improvement)")
print("  2. Challenger F1  >= Champion F1 - 0.01   (no significant recall loss)")
print("  3. Challenger AUC > 0.60 (absolute minimum quality floor)")

In [0]:
# COMMAND ----------
# ── Register the best Session 2 model as the Challenger ───────────────────────

# ── Step 1: Identify the Challenger run (best from S2) ────────────────────────
challenger_run_id = BEST_RUN_ID_S2
challenger_auc    = float(compare_df.iloc[0]["roc_auc"])
challenger_f1     = float(compare_df.iloc[0]["f1"])

print(f"Challenger run ID : {challenger_run_id}")
print(f"Challenger AUC    : {challenger_auc:.4f}")
print(f"Challenger F1     : {challenger_f1:.4f}")

# ── Step 2: Get current Champion (model in Staging from Session 1) ─────────────
# NOTE: Session 1 left the best model in Staging.
# In this session that Staging model IS the Champion
# (it is the model currently closest to production).
try:
    staging_versions = client.get_latest_versions(MODEL_NAME, stages=["Staging"])
    if staging_versions:
        champion_mv      = staging_versions[0]
        champion_version = champion_mv.version
        champion_run     = mlflow.get_run(champion_mv.run_id)
        champion_auc     = float(champion_run.data.metrics.get("roc_auc", 0))
        champion_f1      = float(champion_run.data.metrics.get("f1_score", 0))
        print(f"\nChampion (S1 Staging) version : {champion_version}")
        print(f"Champion AUC                  : {champion_auc:.4f}")
        print(f"Champion F1                   : {champion_f1:.4f}")
    else:
        # No model in Staging — first deployment
        champion_version = None
        champion_auc     = 0.0
        champion_f1      = 0.0
        print("\nNo Staging model found. This will be the first deployment.")
except Exception as e:
    champion_version = None
    champion_auc     = 0.0
    champion_f1      = 0.0
    print(f"Could not retrieve Staging model: {e}")

In [0]:
# COMMAND ----------
# ── Step 3: Run the promotion decision ────────────────────────────────────────

MIN_AUC_IMPROVEMENT = 0.005   # challenger must beat champion by at least this
MAX_F1_REGRESSION   = 0.01    # challenger F1 must not drop by more than this
ABS_MIN_AUC         = 0.60    # hard floor — any model below this is rejected

print("=" * 60)
print("PROMOTION DECISION — Champion vs Challenger")
print("=" * 60)
print(f"\n  {'Metric':<20} {'Champion':>12} {'Challenger':>12}")
print(f"  {'-'*20} {'-'*12} {'-'*12}")
print(f"  {'ROC-AUC':<20} {champion_auc:>12.4f} {challenger_auc:>12.4f}")
print(f"  {'F1 Score':<20} {champion_f1:>12.4f} {challenger_f1:>12.4f}")

# ── Evaluate each promotion criterion ─────────────────────────────────────────
gate_results = {}

# Gate 1: Challenger AUC beats Champion by MIN_AUC_IMPROVEMENT
gate_results["auc_improvement"] = challenger_auc > champion_auc + MIN_AUC_IMPROVEMENT

# Gate 2: Challenger F1 does not regress beyond MAX_F1_REGRESSION
gate_results["f1_no_regression"] = challenger_f1 >= champion_f1 - MAX_F1_REGRESSION

# Gate 3: Absolute quality floor
gate_results["abs_quality_floor"] = challenger_auc >= ABS_MIN_AUC

print(f"\nPromotion Gates:")
print(f"  {'Gate':<35} {'Required':<25} Status")
print(f"  {'-'*35} {'-'*25} ------")
for gate, passed in gate_results.items():
    if gate == "auc_improvement":
        req = f"Challenger AUC > {champion_auc:.4f} + {MIN_AUC_IMPROVEMENT}"
    elif gate == "f1_no_regression":
        req = f"Challenger F1 >= {champion_f1:.4f} - {MAX_F1_REGRESSION}"
    else:
        req = f"Challenger AUC >= {ABS_MIN_AUC}"
    status = "PASS" if passed else "FAIL"
    print(f"  {gate:<35} {req:<25} [{status}]")

all_gates_passed = all(gate_results.values())

print(f"\nFinal decision: {'PROMOTE Challenger to Staging' if all_gates_passed else 'KEEP Champion in Staging'}")

PROMOTION_APPROVED = all_gates_passed

In [0]:
# COMMAND ----------
# ── Step 4: Execute promotion or explain why it was blocked ───────────────────

if PROMOTION_APPROVED:
    # Register the challenger to the Model Registry
    result = mlflow.register_model(
        model_uri=f"runs:/{challenger_run_id}/model",
        name=MODEL_NAME,
    )
    new_version = result.version

    # Add description to the new version
    client.update_model_version(
        name=MODEL_NAME,
        version=new_version,
        description=(
            f"Session 2 Challenger — promoted after beating Champion. "
            f"AUC: {challenger_auc:.4f} (Champion was {champion_auc:.4f}). "
            f"F1: {challenger_f1:.4f}. "
            f"Run ID: {challenger_run_id}. "
            f"Tuned with RandomizedSearchCV 20-iter 5-fold."
        ),
    )

    # Archive old Staging (the Session 1 Champion)
    if champion_version:
        client.transition_model_version_stage(
            name=MODEL_NAME,
            version=champion_version,
            stage="Archived",
            archive_existing_versions=False,
        )
        print(f"Previous Staging (v{champion_version}) archived.")

    # Promote new version to Staging
    client.transition_model_version_stage(
        name=MODEL_NAME,
        version=new_version,
        stage="Staging",
        archive_existing_versions=False,
    )

    CHALLENGER_VERSION = new_version
    print(f"Challenger v{new_version} promoted to Staging.")
    print(f"AUC improvement: {challenger_auc - champion_auc:+.4f}")

else:
    # Challenger did not beat champion — keep current Staging model
    print("Challenger promotion BLOCKED.")
    print("The Champion (Session 1 model) remains in Staging.")
    print("\nNext steps:")
    failed_gates = [g for g, v in gate_results.items() if not v]
    for gate in failed_gates:
        print(f"  Fix required: {gate}")
    # Still register for reference / future use
    result = mlflow.register_model(
        model_uri=f"runs:/{challenger_run_id}/model",
        name=MODEL_NAME,
    )
    CHALLENGER_VERSION = result.version
    client.update_model_version(
        name=MODEL_NAME,
        version=CHALLENGER_VERSION,
        description=(
            f"Session 2 Challenger — NOT promoted (failed gates: {failed_gates}). "
            f"AUC: {challenger_auc:.4f}. F1: {challenger_f1:.4f}. "
            f"Archived for reference."
        ),
    )
    print(f"\nChallenger registered as v{CHALLENGER_VERSION} (no stage — not promoted).")

print(f"\nNavigate to: Models → {MODEL_NAME}")
print("to see all versions and their stages.")

In [0]:
# COMMAND ----------
# ── Promote Staging → Production (if gates passed) ────────────────────────────
# Staging → Production is a separate, deliberate step.
# A model in Staging is validated and approved but not yet live.
# Production is what end-users and downstream systems see.
#
# In a real pipeline this step would require:
#   - A human approval (pull request approval, ops sign-off)
#   - Or a fully automated gate (Session 4 covers this)
#   - An audit trail of who promoted what and when

# Get current Staging version (whatever we just promoted, or the S1 model)
staging_versions = client.get_latest_versions(MODEL_NAME, stages=["Staging"])

if staging_versions:
    staging_v = staging_versions[0]

    # Check if there is already a Production model to archive
    prod_versions = client.get_latest_versions(MODEL_NAME, stages=["Production"])

    if prod_versions:
        old_prod_v = prod_versions[0]
        client.transition_model_version_stage(
            name=MODEL_NAME,
            version=old_prod_v.version,
            stage="Archived",
            archive_existing_versions=False,
        )
        print(f"Previous Production v{old_prod_v.version} → Archived (rollback available).")

    # Promote Staging → Production
    client.transition_model_version_stage(
        name=MODEL_NAME,
        version=staging_v.version,
        stage="Production",
        archive_existing_versions=False,
    )

    PRODUCTION_VERSION = staging_v.version

    print(f"v{staging_v.version} promoted: Staging → Production")
    print()
    print("ROLLBACK INSTRUCTION (if Production model starts behaving badly):")
    if prod_versions:
        print(f"  client.transition_model_version_stage(")
        print(f"      name='{MODEL_NAME}',")
        print(f"      version='{old_prod_v.version}',")
        print(f"      stage='Production'")
        print(f"  )")
        print(f"  — This reverts to v{old_prod_v.version} in seconds with zero downtime.")
    else:
        print("  (No previous Production version to roll back to yet.)")
else:
    print("No Staging model found. Check the previous cell.")

In [0]:
# COMMAND ----------
# ============================================================
# PART 4: Shadow Mode Testing
#
# Promoting a Challenger directly to Production
# based on offline metrics alone is risky. Offline test-set
# performance does not always reflect live traffic behaviour.
#
# Shadow mode = run the new model on real traffic in parallel
# with the current model, but do NOT use its predictions.
# Compare outputs silently for a period (hours / days).
#
# What shadow mode answers:
#   - Do the models agree on most customers?
#   - Where do they DISAGREE? (high-value customers? edge cases?)
#   - Is the Challenger's score distribution similar to Champion's?
#   - Does the Challenger over-predict or under-predict churn?
#
# We simulate this by scoring the same dataset with both the
# Champion (old S1 model, now Archived) and the Challenger
# (new S2 model, now Production) and comparing their outputs.
# ============================================================

print("PART 4 — Shadow Mode Testing")
print("-" * 55)
print("Champion : Session 1 best model (archived, represents old production)")
print("Challenger: Session 2 tuned model (now in Production)")
print()
print("We score the SAME customers with both models and compare:")
print("  - Prediction agreement rate")
print("  - Disagreement analysis (where do they differ?)")
print("  - Score distribution comparison")
print("  - Risk tier shifts")

In [0]:
# COMMAND ----------
# ── Load Champion (S1 best model from Archived stage) ─────────────────────────

import mlflow.sklearn
import mlflow.pyfunc

# Load Champion from Archived stage — the S1 model before it was displaced
# We need to find its exact version
all_versions = client.search_model_versions(f"name='{MODEL_NAME}'")

# S1 best model was the one with the description containing 'Session 1'
s1_version = None
for mv in all_versions:
    if mv.description and "Session 1" in mv.description:
        s1_version = mv.version
        break

if s1_version:
    champion_uri = f"models:/{MODEL_NAME}/{s1_version}"
    champion_model = mlflow.sklearn.load_model(champion_uri)
    print(f"Champion loaded from: {champion_uri}")
else:
    # Fallback: load from any Archived version
    archived = [v for v in all_versions if v.current_stage == "Archived"]
    if archived:
        s1_version = archived[-1].version
        champion_uri = f"models:/{MODEL_NAME}/{s1_version}"
        champion_model = mlflow.sklearn.load_model(champion_uri)
        print(f"Champion loaded from Archived: {champion_uri}")
    else:
        print("WARNING: No archived S1 model found.")
        print("Using the current Production model as both Champion and Challenger for demo.")
        champion_model = best_model_s2

# ── Load Challenger (Production) ─────────────────────────────────────────────
challenger_model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/Production")
print(f"Challenger loaded from: models:/{MODEL_NAME}/Production")

In [0]:
# COMMAND ----------
# ── Score ALL customers with both Champion and Challenger ─────────────────────

import pandas as pd
import numpy as np

X_all   = df_model[FEATURE_COLS]
cust_ids = df_model["CustomerID"].values
true_labels = df_model[TARGET_COL].values

# Score with Champion
champ_probs  = champion_model.predict_proba(X_all)[:, 1]
champ_preds  = champion_model.predict(X_all)

# Score with Challenger
chall_probs  = challenger_model.predict_proba(X_all)[:, 1]
chall_preds  = challenger_model.predict(X_all)

# ── Build shadow comparison table ─────────────────────────────────────────────
shadow_df = pd.DataFrame({
    "CustomerID":         cust_ids,
    "true_churn":         true_labels,
    "champion_prob":      champ_probs,
    "champion_pred":      champ_preds,
    "challenger_prob":    chall_probs,
    "challenger_pred":    chall_preds,
    "agreement":          (champ_preds == chall_preds).astype(int),
    "prob_delta":         chall_probs - champ_probs,  # positive = challenger scores higher
})

# Risk tier labels
def risk_tier(prob):
    if prob >= 0.60:   return "HIGH"
    elif prob >= 0.30: return "MEDIUM"
    else:              return "LOW"

shadow_df["champion_tier"]   = shadow_df["champion_prob"].apply(risk_tier)
shadow_df["challenger_tier"] = shadow_df["challenger_prob"].apply(risk_tier)
shadow_df["tier_changed"]    = (shadow_df["champion_tier"] != shadow_df["challenger_tier"]).astype(int)

# Save to Delta
spark.createDataFrame(shadow_df).write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(SCORES_PATH_S2)

print(f"Shadow scores saved to Delta: {SCORES_PATH_S2}")
print(f"Total customers scored: {len(shadow_df):,}")

In [0]:
# COMMAND ----------
# ── Shadow mode analysis ──────────────────────────────────────────────────────

n_total   = len(shadow_df)
n_agree   = shadow_df["agreement"].sum()
n_disagree = n_total - n_agree
n_tier_changed = shadow_df["tier_changed"].sum()

print("=" * 60)
print("SHADOW MODE ANALYSIS")
print("=" * 60)

print(f"\nPrediction agreement:")
print(f"  Total customers  : {n_total:,}")
print(f"  Agree            : {n_agree:,}  ({n_agree/n_total:.1%})")
print(f"  Disagree         : {n_disagree:,}  ({n_disagree/n_total:.1%})")
print(f"  Risk tier changed: {n_tier_changed:,}  ({n_tier_changed/n_total:.1%})")

# ── Disagreement breakdown ────────────────────────────────────────────────────
disagree_df = shadow_df[shadow_df["agreement"] == 0]

# Champion says RETAIN, Challenger says CHURN (challenger catches more)
champ_retain_chall_churn = disagree_df[
    (disagree_df["champion_pred"] == 0) & (disagree_df["challenger_pred"] == 1)
]
# Champion says CHURN, Challenger says RETAIN (challenger is less aggressive)
champ_churn_chall_retain = disagree_df[
    (disagree_df["champion_pred"] == 1) & (disagree_df["challenger_pred"] == 0)
]

print(f"\nDisagreement types:")
print(f"  Champion=RETAIN, Challenger=CHURN : {len(champ_retain_chall_churn):>4,}  (Challenger more aggressive)")
print(f"  Champion=CHURN,  Challenger=RETAIN: {len(champ_churn_chall_retain):>4,}  (Challenger less aggressive)")

# ── Score distribution comparison ────────────────────────────────────────────
print(f"\nScore distribution (churn probability):")
print(f"  {'Percentile':<12} {'Champion':>12} {'Challenger':>12} {'Delta':>10}")
print(f"  {'-'*12} {'-'*12} {'-'*12} {'-'*10}")
for pct in [25, 50, 75, 90, 95]:
    c_val  = np.percentile(champ_probs, pct)
    ch_val = np.percentile(chall_probs, pct)
    print(f"  {f'p{pct}':<12} {c_val:>12.4f} {ch_val:>12.4f} {ch_val-c_val:>+10.4f}")

# ── Risk tier comparison ──────────────────────────────────────────────────────
print(f"\nRisk tier distribution:")
print(f"  {'Tier':<10} {'Champion':>12} {'Challenger':>12}")
for tier in ["HIGH", "MEDIUM", "LOW"]:
    c_n  = (shadow_df["champion_tier"]   == tier).sum()
    ch_n = (shadow_df["challenger_tier"] == tier).sum()
    print(f"  {tier:<10} {c_n:>12,} {ch_n:>12,}")

# ── Accuracy on customers where we have true labels ───────────────────────────
from sklearn.metrics import roc_auc_score
champ_full_auc = roc_auc_score(true_labels, champ_probs)
chall_full_auc = roc_auc_score(true_labels, chall_probs)

print(f"\nFull-dataset AUC (all 613 customers):")
print(f"  Champion AUC   : {champ_full_auc:.4f}")
print(f"  Challenger AUC : {chall_full_auc:.4f}")
print(f"  Improvement    : {chall_full_auc - champ_full_auc:+.4f}")

# ── Shadow mode verdict ───────────────────────────────────────────────────────
print(f"\nShadow Mode Verdict:")
print("-" * 60)
if chall_full_auc > champ_full_auc:
    print("  CONFIRM: Challenger performs better on full dataset.")
    print("  Action : Keep Challenger in Production. Monitor for 7 days.")
else:
    print("  CAUTION: Challenger does NOT outperform Champion on full dataset.")
    print("  Action : Consider rolling back to Champion.")
    print("  Command: client.transition_model_version_stage(")
    print(f"               name='{MODEL_NAME}', version='{s1_version}', stage='Production')")

In [0]:
# COMMAND ----------
# ── Log shadow mode results back into MLflow ──────────────────────────────────
# Log operational results (not just training metrics).
# Shadow mode outcomes are as important as offline metrics for deciding
# whether to keep or revert a deployment.

with mlflow.start_run(run_name="shadow_mode_analysis") as shadow_run:

    mlflow.log_params({
        "champion_model":      f"{MODEL_NAME}/v{s1_version}",
        "challenger_model":    f"{MODEL_NAME}/Production",
        "customers_scored":    n_total,
        "data_version":        DATA_VERSION,
    })

    mlflow.log_metrics({
        "agreement_rate":         round(n_agree / n_total, 4),
        "disagreement_rate":      round(n_disagree / n_total, 4),
        "tier_change_rate":       round(n_tier_changed / n_total, 4),
        "champion_full_auc":      round(champ_full_auc, 4),
        "challenger_full_auc":    round(chall_full_auc, 4),
        "auc_delta":              round(chall_full_auc - champ_full_auc, 4),
        "champ_retain_chall_churn": len(champ_retain_chall_churn),
        "champ_churn_chall_retain": len(champ_churn_chall_retain),
    })

    mlflow.set_tags({
        "session":    "session-2",
        "run_type":   "shadow_mode",
        "team":       "mlops-training",
        "verdict":    "CONFIRMED" if chall_full_auc > champ_full_auc else "ROLLBACK_ADVISED",
    })

    shadow_run_id = shadow_run.info.run_id

print(f"Shadow mode results logged to run: {shadow_run_id}")
print("Navigate to MLflow → mlops_s2_churn → shadow_mode_analysis to review.")

In [0]:
# COMMAND ----------
# ── Print complete Model Registry state ───────────────────────────────────────
# Always end a session with a clear view of what is registered and where.

print("=" * 60)
print(f"MODEL REGISTRY STATE — {MODEL_NAME}")
print("=" * 60)

all_mv = sorted(
    client.search_model_versions(f"name='{MODEL_NAME}'"),
    key=lambda x: int(x.version)
)

print(f"\n  {'Version':<10} {'Stage':<14} {'Description (truncated)'}")
print(f"  {'-'*10} {'-'*14} {'-'*40}")
for mv in all_mv:
    desc_short = (mv.description or "")[:55] + "..." if mv.description and len(mv.description) > 55 else (mv.description or "")
    print(f"  v{mv.version:<9} {mv.current_stage:<14} {desc_short}")

In [0]:
# COMMAND ----------
# ============================================================
# SESSION 2 COMPLETE — Summary
# ============================================================

print("=" * 60)
print("SESSION 2 — COMPLETE")
print("=" * 60)

print("\nWHAT YOU LEARNED vs Session 1:")
print("-" * 60)
lessons = [
    ("S1: 3 manual configs tried",      "S2: RandomizedSearchCV over distributions"),
    ("S1: Single train/test eval",       "S2: 5-fold cross-validation per config"),
    ("S1: Flat MLflow runs",             "S2: Nested runs (parent + 20 child trials)"),
    ("S1: Only metrics logged",          "S2: Confusion matrix + ROC curve as artifacts"),
    ("S1: Manual run comparison",        "S2: Programmatic search_runs() across sessions"),
    ("S1: No promotion gate",            "S2: 3-gate Champion/Challenger comparison"),
    ("S1: Register + stage in one step", "S2: Register → Staging → Production separately"),
    ("S1: No rollback plan",             "S2: Explicit rollback command documented"),
    ("S1: No production validation",     "S2: Shadow mode on full dataset post-deploy"),
    ("S1: No shadow mode",               "S2: Agreement rate, disagreement analysis, verdict"),
]
for s1, s2 in lessons:
    print(f"  {s1:<35} → {s2}")


